In [5]:
!pip install -q pypdf langchain langchain-community langchain-groq sentence-transformers chromadb langchain-text-splitters

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Load the PDF
file_path = "/content/Seerat e Mustafa_new.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
full_text = " ".join([doc.page_content for doc in documents])

print(f"Loaded PDF with {len(documents)} pages.")

# 1. Fixed-size Chunking (No overlap)
fixed_splitter = CharacterTextSplitter(separator=" ", chunk_size=500, chunk_overlap=0)
fixed_chunks = fixed_splitter.create_documents([full_text])

# 2. Overlapping Chunking
overlap_splitter = CharacterTextSplitter(separator=" ", chunk_size=500, chunk_overlap=100)
overlap_chunks = overlap_splitter.create_documents([full_text])

# 3. Recursive/Hierarchical Chunking
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", " ", ""])
recursive_chunks = recursive_splitter.create_documents([full_text])

print(f"\n--- Chunking Results ---")
print(f"Fixed-size strategy: {len(fixed_chunks)} chunks")
print(f"Overlapping strategy: {len(overlap_chunks)} chunks")
print(f"Recursive strategy: {len(recursive_chunks)} chunks")

/tmp/ipykernel_445/3847916706.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded PDF with 675 pages.

--- Chunking Results ---
Fixed-size strategy: 3186 chunks
Overlapping strategy: 3951 chunks
Recursive strategy: 3498 chunks


```markdown
### Hit-Rate@5 Performance Setup
To compare performance, we would typically use a set of questions (ground truth) and check if the retrieved chunks contain the answer. Below is a skeleton using `langchain-groq` and a vector store to simulate how you would evaluate retrieval.

*Note: You will need to provide your `GROQ_API_KEY` in the Colab secrets.*
```

In [ ]:
import torch
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Detect if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Setup Embeddings with GPU support
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': device}
)

def evaluate_strategy(chunks, name):
    # This function is now a helper to create the retriever
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    print(f"Strategy {name} is indexed.")
    return retriever

In [9]:
import pandas as pd
from langchain_groq import ChatGroq
from google.colab import userdata
from tqdm.auto import tqdm
from langchain_community.vectorstores import Chroma
import time

# Note: Ensure GROQ_API_KEY is in your Colab Secrets
try:
    groq_api_key = userdata.get("GROQ_API_KEY")
except:
    groq_api_key = None
    print("Warning: GROQ_API_KEY not found in secrets.")

# Define evaluation queries
eval_queries = [
    "Who is the author of this book?",
    "What are the primary topics covered in the first chapter?",
    "Explain the significance of the title.",
    "Where was the Prophet born?",
    "What is the name of the author?"
]

def calculate_hit_rate(retriever, queries):
    hits = 0
    for query in tqdm(queries, desc="    Evaluating Queries", leave=False):
        results = retriever.invoke(query)
        if len(results) > 0:
            hits += 1
    return (hits / len(queries)) * 100

# Use strategies defined in cell 01f7abbd
strategies = {
    "Fixed-size": fixed_chunks,
    "Overlapping": overlap_chunks,
    "Recursive": recursive_chunks
}

results_data = []

for name, chunks in tqdm(strategies.items(), desc="Processing Strategies"):
    print(f"\nIndexing {len(chunks)} chunks for {name}...")
    # embeddings is defined in cell 453d5983
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    hr5 = calculate_hit_rate(retriever, eval_queries)
    results_data.append({
        "Strategy": name,
        "Hit-Rate@5": f"{hr5:.2f}%",
        "Total Chunks": len(chunks)
    })

df_results = pd.DataFrame(results_data)
print("\n--- Performance Comparison ---")
display(df_results)

/tmp/ipykernel_445/2822527418.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

    Evaluating Queries:   0%|          | 0/5 [00:00<?, ?it/s]


Indexing 3498 chunks for Recursive...


    Evaluating Queries:   0%|          | 0/5 [00:00<?, ?it/s]


--- Performance Comparison ---


,Strategy,Hit-Rate@5,Total Chunks
0,Fixed-size,100.00%,3186
1,Overlapping,100.00%,3951
2,Recursive,100.00%,3498
